In [ ]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
import pandas as pd,os,json

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5-20251001"


df = pd.read_csv(r"telehealth.csv")

def get_df_info():
    temp_df = df.copy()
    temp_df.fillna("N/A", inplace=True)

    return {
        "columns": temp_df.columns.tolist(),
        "rows": len(temp_df),
        "data_types": temp_df.dtypes.astype(str).to_dict(),
        "top_5_rows": temp_df.head().to_dict(orient="records"),
    }

def run_cohort_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        
        total_patients=len(group)
        
        active_patients=(
            group["retention_status"].astype(str).str.lower().eq("active").sum()
        )
        
        churned_patients=total_patients-active_patients
        
        retention_rate=round((active_patients/total_patients)*100,2)
        
        result[program]={
            "number_of_patients":total_patients,
            "average_treatment_duration_weeks":round(group["treatment_duration_weeks"].mean(),2),
            "active_patients":int(active_patients),
            "churned_patients":int(churned_patients),
            "retention_rate_percent":retention_rate,
        }

    return result

def run_outcome_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        retention_rate = round(
            (group["retention_status"].astype(str).str.lower().eq("active").sum() / len(group)) * 100, 2
        )
        duration_q1 = round(group["treatment_duration_weeks"].quantile(0.25), 2)
        duration_median = round(group["treatment_duration_weeks"].quantile(0.5), 2)
        duration_q3 = round(group["treatment_duration_weeks"].quantile(0.75), 2)
        duration_mean = round(group["treatment_duration_weeks"].mean(), 2)
        
        # FIX #1: Only count churned/inactive patients for drop-off analysis
        churned = group[group["retention_status"].astype(str).str.lower().eq("inactive")]
        
        early_dropoff = len(churned[churned["treatment_duration_weeks"] <= duration_q1])
        mid_dropoff = len(churned[(churned["treatment_duration_weeks"] > duration_q1) & 
                                (churned["treatment_duration_weeks"] <= duration_q3)])
        late_dropoff = len(churned[churned["treatment_duration_weeks"] > duration_q3])
        
        result[program] = {
            "retention_rate_percent": retention_rate,
            "duration_trends": {
                "mean_weeks": duration_mean,
                "median_weeks": duration_median,
                "q1_weeks": duration_q1,
                "q3_weeks": duration_q3,
            },
            "drop_off_points": {
                "early_stage_patients": int(early_dropoff),
                "mid_stage_patients": int(mid_dropoff),
                "late_stage_patients": int(late_dropoff),
            }
        }
    
    return result

def flag_anomalies(df):
    result = {}
    
    for program, group in df.groupby("program_type"):
        q1 = group["treatment_duration_weeks"].quantile(0.25)
        q3 = group["treatment_duration_weeks"].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        
        short_duration_outliers = group[group["treatment_duration_weeks"] < lower_bound]
        short_duration_count = len(short_duration_outliers)
        
        churned = group[group["retention_status"].astype(str).str.lower().eq("inactive")]
        churn_rate = round((len(churned) / len(group)) * 100, 2)
        
        abnormal_churn = churn_rate > 25.0
        
        result[program] = {
            "short_duration_outliers": {
                "count": int(short_duration_count),
                "threshold_weeks": round(lower_bound, 2),
                "affected_patients_percent": round((short_duration_count / len(group)) * 100, 2),
                "flagged": short_duration_count > 0,
            },
            "abnormal_drop_off_clusters": {
                "churn_rate_percent": churn_rate,
                "flagged": abnormal_churn,
                "interpretation": "High churn detected" if abnormal_churn else "Normal churn levels"
            }
        }
    
    return result


tools = [
    {
        "name": "get_df_info",
        "description": "Returns dataframe metadata: columns, row count, data types, and first five rows. Use this to understand the dataset structure before any analysis.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_cohort_analysis",
        "description": "Compares programs side-by-side with static snapshots: total patients, active patients, churned patients, retention rate %, and average duration. Use ONLY for comparisons, rankings, and group-level counts. Do not use for analyzing trends, timing, or drop-off patterns.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_outcome_analysis",
        "description": "Analyzes patient engagement over time: retention rates paired with duration trends (quartiles), early/mid/late stage drop-off patterns, and temporal churn distribution. Use for understanding WHEN patients drop off and HOW treatment duration varies. Do not use for simple retention comparisons—use cohort_analysis for that.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "flag_anomalies",
        "description": "Identifies outliers and exceptions: unusually short treatment durations (IQR method) and abnormal churn rates (>25%). Flags risk and data quality issues. Use when the user wants to identify risk signals, data anomalies, or unusual patterns that require investigation or intervention.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

def add_user_message(messages, message):
    messages.append(
        {
            "role": "user",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def add_assistant_message(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def chat(messages, system=None, temperature=0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "tools": tools,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    return client.messages.create(**params)


def text_from_message(message):
    texts = []

    for block in message.content:
        if block.type == "text":
            texts.append(block.text)

    return "\n".join(texts)


def run_tool(tool_name, tool_input):
    if tool_name == "get_df_info":
        return get_df_info()
    elif tool_name =='run_cohort_analysis':
        return run_cohort_analysis(df)
    elif tool_name == 'run_outcome_analysis':
        return run_outcome_analysis(df)
    elif tool_name == 'flag_anomalies':
        return flag_anomalies(df)

    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_results = []

    for block in message.content:

        if block.type != "tool_use":
            continue

        try:
            result = run_tool(block.name, block.input)

            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

        except Exception as e:
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(e),
                    "is_error": True,
                }
            )

    return tool_results


def run_conversation(messages):
    tools_used = []
    
    while True:
        response = chat(messages)
        
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_requests = [block for block in response.content if block.type == "tool_use"]
        for tool_request in tool_requests:
            tools_used.append(tool_request.name)

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages, tools_used


messages = []
add_user_message(
    messages,
    "Analyze the telehealth patient dataset and provide insights on patient retention and treatment outcomes across different programs. Highlight any anomalies or outliers in the data."
)

messages, tools_used = run_conversation(messages)

output = {
    "tools_used": list(set(tools_used)),
    "tool_call_count": len(tools_used),
    "message_history_length": len(messages),
    "messages": messages
}

print("\n" + "="*80)
print("TOOLS USED:")
print("="*80)
for tool in set(tools_used):
    count = tools_used.count(tool)
    print(f"  {tool} (called {count} time{'s' if count > 1 else ''})")

print("\n" + "="*80)
print("OUTPUT (JSON FORMAT):")
print("="*80 + "\n")
print(json.dumps(output, indent=2, default=str))